# Session 7: Tiny Quantum Transformer Block

In this session, we build the first trainable Quantum Transformer-style block.

The goal is to combine the quantum attention concepts explored in Session 2 into a model that can be trained and evaluated on a classification task.

In [1]:
import numpy as np

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.utils.loss_functions import CrossEntropyLoss

from qiskit_algorithms.optimizers import COBYLA

## Dataset

In [2]:
X, y = make_moons(
    n_samples=400,
    noise=0.15,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train_q = 2 * y_train - 1
y_test_q = 2 * y_test - 1

print("Training Samples:", len(X_train))
print("Test Samples:", len(X_test))

Training Samples: 320
Test Samples: 80


##  Quantum Query, Key, and Value Circuits

In [3]:
num_qubits = 2

x_params = ParameterVector("x", 2)

q_weights = ParameterVector("q", 2)
k_weights = ParameterVector("k", 2)
v_weights = ParameterVector("v", 2)

In [4]:
def create_qkv_circuit(weight_params):

    qc = QuantumCircuit(num_qubits)

    qc.ry(x_params[0], 0)
    qc.ry(x_params[1], 1)

    qc.ry(weight_params[0], 0)
    qc.ry(weight_params[1], 1)

    qc.cx(0, 1)

    return qc

In [5]:
query_circuit = create_qkv_circuit(q_weights)
key_circuit = create_qkv_circuit(k_weights)
value_circuit = create_qkv_circuit(v_weights)

print("Query Circuit")
print(query_circuit)

Query Circuit
     ┌──────────┐┌──────────┐     
q_0: ┤ Ry(x[0]) ├┤ Ry(q[0]) ├──■──
     ├──────────┤├──────────┤┌─┴─┐
q_1: ┤ Ry(x[1]) ├┤ Ry(q[1]) ├┤ X ├
     └──────────┘└──────────┘└───┘


In [6]:
sample = X_train[0]

q_weights_values = np.random.rand(2)
k_weights_values = np.random.rand(2)
v_weights_values = np.random.rand(2)

print("Input Sample:")
print(sample)

print("\nQ Weights:")
print(q_weights_values)

print("\nK Weights:")
print(k_weights_values)

print("\nV Weights:")
print(v_weights_values)

Input Sample:
[-0.30926812 -0.01326385]

Q Weights:
[0.58538879 0.29095665]

K Weights:
[0.43346029 0.36620786]

V Weights:
[0.7306494  0.99440291]


In [7]:
query_params = np.concatenate(
    [sample, q_weights_values]
)

key_params = np.concatenate(
    [sample, k_weights_values]
)

value_params = np.concatenate(
    [sample, v_weights_values]
)

print("Query Parameters:")
print(query_params)

print("\nKey Parameters:")
print(key_params)

print("\nValue Parameters:")
print(value_params)

Query Parameters:
[-0.30926812 -0.01326385  0.58538879  0.29095665]

Key Parameters:
[-0.30926812 -0.01326385  0.43346029  0.36620786]

Value Parameters:
[-0.30926812 -0.01326385  0.7306494   0.99440291]


## Compute Query, Key, and Value Outputs

In [8]:
from qiskit.quantum_info import Statevector

In [10]:
query_state = Statevector.from_instruction(
    query_circuit.assign_parameters(
        {
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            q_weights[0]: q_weights_values[0],
            q_weights[1]: q_weights_values[1]
        }
    )
)

key_state = Statevector.from_instruction(
    key_circuit.assign_parameters(
        {
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            k_weights[0]: k_weights_values[0],
            k_weights[1]: k_weights_values[1]
        }
    )
)

value_state = Statevector.from_instruction(
    value_circuit.assign_parameters(
        {
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            v_weights[0]: v_weights_values[0],
            v_weights[1]: v_weights_values[1]
        }
    )
)

print("Query State:")
print(query_state.data)

print("\nKey State:")
print(key_state.data)

print("\nValue State:")
print(value_state.data)

Query State:
[0.98095269+0.j 0.01904701+0.j 0.1370838 +0.j 0.13629773+0.j]

Key State:
[0.98257177+0.j 0.01089443+0.j 0.17521911+0.j 0.0610924 +0.j]

Value State:
[0.86255933+0.j 0.09852957+0.j 0.46071013+0.j 0.18447088+0.j]


## Attention Score

In [11]:
attention_score = np.abs(
    np.vdot(
        query_state.data,
        key_state.data
    )
)

print("Attention Score:")
print(attention_score)

Attention Score:
0.9964103850536329


In [12]:
attention_output = (
    attention_score *
    value_state.data
)

print("Attention Output:")
print(attention_output)

Attention Output:
[0.85946307+0.j 0.09817589+0.j 0.45905636+0.j 0.1838087 +0.j]


In [13]:
attention_features = np.real(
    attention_output
)

print("Attention Features:")
print(attention_features)

print("Feature Shape:")
print(attention_features.shape)

Attention Features:
[0.85946307 0.09817589 0.45905636 0.1838087 ]
Feature Shape:
(4,)


In [14]:
transformer_feature = np.concatenate([
    attention_features
])

print("Transformer Representation:")
print(transformer_feature)

print("Representation Shape:")
print(transformer_feature.shape)

Transformer Representation:
[0.85946307 0.09817589 0.45905636 0.1838087 ]
Representation Shape:
(4,)


## Transformer Representations for the Dataset

In [15]:
def quantum_attention_embedding(sample):

    q_w = np.random.rand(2)
    k_w = np.random.rand(2)
    v_w = np.random.rand(2)

    query_state = Statevector.from_instruction(
        query_circuit.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            q_weights[0]: q_w[0],
            q_weights[1]: q_w[1]
        })
    )

    key_state = Statevector.from_instruction(
        key_circuit.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            k_weights[0]: k_w[0],
            k_weights[1]: k_w[1]
        })
    )

    value_state = Statevector.from_instruction(
        value_circuit.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            v_weights[0]: v_w[0],
            v_weights[1]: v_w[1]
        })
    )

    attention_score = np.abs(
        np.vdot(
            query_state.data,
            key_state.data
        )
    )

    attention_output = (
        attention_score *
        np.real(value_state.data)
    )

    return attention_output

In [16]:
X_train_transformer = np.array([
    quantum_attention_embedding(x)
    for x in X_train
])

X_test_transformer = np.array([
    quantum_attention_embedding(x)
    for x in X_test
])

print("Train Shape:", X_train_transformer.shape)
print("Test Shape :", X_test_transformer.shape)

Train Shape: (320, 4)
Test Shape : (80, 4)


## Training Classification Head

In [17]:
from sklearn.linear_model import LogisticRegression

In [18]:
classifier = LogisticRegression()

classifier.fit(
    X_train_transformer,
    y_train
)

print("Classifier trained.")

Classifier trained.


In [19]:
accuracy = classifier.score(
    X_test_transformer,
    y_test
)

print("Tiny Quantum Transformer Accuracy:")
print(accuracy)

Tiny Quantum Transformer Accuracy:
0.825
